# Data Mining - Lab 9
## Cube space: Data Cube-ийн тооцоолол ба олон хэмжээст шинжилгээ

**Dataset:** `cs_coffee_mini.csv`, `cs_sales.csv`, `cs_dim_item.csv`, `cs_dim_location.csv`, `cs_iceberg_demo.csv`, `cs_closed_demo.csv`, `cs_monthly_item_city.csv`, `cs_loans.csv`

Энэ лаб нь "Cube space: олон хэмжээст өгөгдлийн шинжилгээ" лекцийн агуулгад үндэслэсэн: cuboid, lattice, cell, concept hierarchy, measure, cube-ийн тооцоолол (materialization, iceberg, BUC, closed cell), OLAP архитектур, cube space дахь mining (prediction cube, discovery-driven exploration).

| Файл | Мөр | Тайлбар |
|---|---|---|
| `cs_coffee_mini` | 8 | Лекцийн кофе шопын мини куб (салбар x улирал x бараа) |
| `cs_sales` | 3000 | 2026 оны 1-6 сарын электроникийн борлуулалтын гүйлгээ |
| `cs_dim_item` | 8 | Барааны dimension (төрөл, брэнд, нийлүүлэгчийн төрөл, нэгж үнэ) |
| `cs_dim_location` | 9 | Байршлын dimension (дүүрэг, хот, улс) |
| `cs_iceberg_demo` | 10 | Лекцийн Iceberg/BUC дасгалын бичлэгүүд |
| `cs_closed_demo` | 5 | Closed cell ба quotient cube-ийн жижиг жишээ |
| `cs_monthly_item_city` | 168 | Бараа x сар x хотын борлуулалт (`month = 0` нь 2025 оны 12-р сар, суурь) |
| `cs_loans` | 2370 | Зээлийн шийдвэрийн өгөгдөл (хот x жил), prediction cube-д |

### Lab зорилго
- Cell (base, aggregate, apex), ancestor ба descendant ойлголтыг кодоор ялгах
- Cuboid-ийн lattice, шатлалтай cube-ийн cuboid-ийн тоог тооцох, `GROUP BY CUBE`-ийг хэрэгжүүлэх
- Concept hierarchy (schema ба set grouping) ашиглан roll-up, drill-down хийх
- Distributive, algebraic, holistic measure-ийн нэгтгэлийн дүрмийг шалгах
- Cuboid сонгох, materialization, Iceberg cube, BUC, closed cell, quotient cube-ийг хэрэгжүүлэх
- ROLAP ба MOLAP-ийг харьцуулах
- Discovery-driven exploration ба prediction cube-ээр cube space дотор mining хийх

Даалгавар бүр **TASK** гэсэн гарчигтай. Код бүрэн өгөгдөөгүй: функцийн нэр, хувьсагчийн нэр, `# TODO` маягийн заавар өгсөн тул `...` болон тайлбарын дагуу кодыг **өөрөө бичнэ**. Хариугаа багшид үзүүлж шалгуулна.

## 0) Өгөгдөл унших

Гүйлгээний хүснэгтийг `cs_dim_item`, `cs_dim_location`-тэй холбож нэг өргөн хүснэгт (`df`) гаргана. Үүн дээр сарын түвшинг (`month`) нэмнэ.

In [1]:
import pandas as pd
import numpy as np
import sqlite3, math
from itertools import combinations

sales    = pd.read_csv("cs_sales.csv")
dim_item = pd.read_csv("cs_dim_item.csv")
dim_loc  = pd.read_csv("cs_dim_location.csv")

df = (sales.merge(dim_item, on="item", validate="many_to_one")
           .merge(dim_loc, on="district", validate="many_to_one"))
df["date"]  = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.strftime("%Y-%m")
print(df.shape, df["sales"].sum())
df.head()

(3000, 16) 2602025


,sale_id,date,item,district,customer_group,units,sales,item_key,type,brand,supplier_type,unit_price,location_key,city,country,month
0,1,2026-01-01,Гар,Чаоян,Хувь хүн,4,200,7,Дагалдах хэрэгсэл,Монгол Тех,Дотоодын,50,7,Бээжин,Хятад,2026-01
1,2,2026-01-01,Монитор,Баянзүрх,Бизнес,2,600,2,Компьютер,Dell,Гадаадын,300,3,Улаанбаатар,Монгол,2026-01
2,3,2026-01-01,Гар,Дархан-Уул,Хувь хүн,1,50,7,Дагалдах хэрэгсэл,Монгол Тех,Дотоодын,50,4,Дархан,Монгол,2026-01
3,4,2026-01-01,Хулгана,Хан-Уул,Хувь хүн,5,125,8,Дагалдах хэрэгсэл,Монгол Тех,Дотоодын,25,2,Улаанбаатар,Монгол,2026-01
4,5,2026-01-01,Утас,Чаоян,Хувь хүн,1,400,4,Гар утас ба таблет,Samsung,Гадаадын,400,7,Бээжин,Хятад,2026-01


## 1) Мини куб: Roll-up, slice, dice (лекцийн дасгал)

Кофе шопын борлуулалт (мянган ₮). Dimension: `branch`, `quarter`, `item`. Measure: `sales`.

### TASK 1: Мини кубийн 6 асуулт

1. **Roll-up**: салбар бүрийн нийт борлуулалт.
2. **Slice**: `quarter = Q2` хүснэгтийг (мөр = салбар, багана = бараа) гарга.
3. **Dice**: `branch = УБ` ба `item = Кофе`. Улирал бүрийн утгыг гарга.
4. **Apex**: `(*, *, *)` cell-ийн утга.
5. **Cuboid**: энэ кубэд хэдэн cuboid байх вэ?
6. **Exception**: Q1-ээс Q2 хүртэл хамгийн их хувиар өссөн (салбар, бараа) cell аль вэ?

In [2]:
coffee = pd.read_csv("cs_coffee_mini.csv")
print(coffee)

# 1. Roll-up: салбар бүрийн нийт борлуулалт
print(coffee.groupby("branch")["sales"].sum())

# 2. Slice: quarter == "Q2" хүснэгт (мөр = салбар, багана = бараа), pivot_table ашигла
q2 = coffee[coffee["quarter"] == "Q2"].pivot_table(index="branch", columns="item",
                                                  values="sales", aggfunc="sum")
print(q2)

# 3. Dice: branch == "УБ" ба item == "Кофе", улирал бүрийн утга
dice = coffee[(coffee["branch"] == "УБ") & (coffee["item"] == "Кофе")]
print(dice[["quarter", "sales"]])

# 4. Apex-ийн утга ба cuboid-ийн тоо (2 ** n)
print("apex (*,*,*) =", coffee["sales"].sum(), "| cuboid =", 2 ** 3)

# 5. Exception: Q1 -> Q2 өсөлтийг хувиар бодож, хамгийн их өссөн (салбар, бараа)-г ол
#    Санамж: coffee.pivot_table(index=["branch", "item"], columns="quarter", values="sales")
pv = coffee.pivot_table(index=["branch", "item"], columns="quarter", values="sales")
pv["growth"] = (pv["Q2"] - pv["Q1"]) / pv["Q1"] * 100
print(pv.round(1).sort_values("growth", ascending=False))
print("хамгийн их өсөлт:", pv["growth"].idxmax(), round(pv["growth"].max(), 1), "%")


   branch quarter  item  sales
0      УБ      Q1  Кофе    120
1      УБ      Q1   Цай     60
2      УБ      Q2  Кофе    150
3      УБ      Q2   Цай     70
4  Дархан      Q1  Кофе     50
5  Дархан      Q1   Цай     40
6  Дархан      Q2  Кофе     65
7  Дархан      Q2   Цай     45
branch
Дархан    200
УБ        400
Name: sales, dtype: int64
item    Кофе  Цай
branch           
Дархан    65   45
УБ       150   70
  quarter  sales
0      Q1    120
2      Q2    150
apex (*,*,*) = 600 | cuboid = 8
quarter         Q1     Q2  growth
branch item                      
Дархан Кофе   50.0   65.0    30.0
УБ     Кофе  120.0  150.0    25.0
       Цай    60.0   70.0    16.7
Дархан Цай    40.0   45.0    12.5
хамгийн их өсөлт: ('Дархан', 'Кофе') 30.0 %


## 2) Cell: base, aggregate, ancestor, descendant

Cell нь cuboid доторх нэг мөр. `*` нь тухайн dimension-ийг нэгтгэсэн (all) гэсэн тэмдэг.

| Cell | Төрөл |
|---|---|
| `(Jan, Chicago, Business)` | base cell (бүх dimension утгатай) |
| `(Jan, *, Business)` | 2-D aggregate cell |
| `(*, *, *)` | apex cell (0-D) |

**Ancestor** (өвөг) нь илүү ерөнхий, **descendant** (үр удам) нь илүү нарийвчилсан cell. `(Jan, *, *)` нь `(Jan, Chicago, Business)`-ийн ancestor.

### TASK 2: Бүх cell-ийг гаргаж ангилах

1. `cube_cells` функцээр мини кубийн **бүх cell** (base, aggregate, apex)-ийг гарга. Нийт cell-ийн тоо `(n₁+1)(n₂+1)(n₃+1)` томьёотой тэнцэх ёстой.
2. `classify` функцийг бөглөж лекцийн Жишээ 3.9-ийн 5 cell-ийг ангил.
3. `is_ancestor` функцээр `(УБ, Q1, Кофе)` cell-ийн ancestor-уудыг, `(УБ, *, *)` cell-ийн descendant-уудыг ол.

In [3]:
dims = ["branch", "quarter", "item"]

def cube_cells(df, dims, measure):
    """Бүх cell (base, aggregate, apex)-ийг нэг DataFrame болгож буцаа.
    Нэгтгэгдсэн dimension-д "*" тавина. Баганууд: dims + [measure]."""
    parts = []
    for k in range(len(dims), -1, -1):
        for sub in combinations(dims, k):
            if sub:
                g = df.groupby(list(sub), as_index=False)[measure].sum()
            else:
                g = pd.DataFrame({measure: [df[measure].sum()]})
            for d in dims:
                if d not in sub:
                    g[d] = "*"
            parts.append(g[dims + [measure]])
    return pd.concat(parts, ignore_index=True)

cells = cube_cells(coffee, dims, "sales")
# Нийт cell-ийн тоог (n1 + 1)(n2 + 1)(n3 + 1) томьёотой харьцуул
expected = math.prod(coffee[d].nunique() + 1 for d in dims)
print("cell:", len(cells), "| томьёо:", expected, "|", len(cells) == expected)

def classify(cell):
    """cell-ийг "base cell", "apex cell" эсвэл "k-D aggregate cell" гэж ангил."""
    k = sum(v != "*" for v in cell)
    if k == len(cell):
        return "base cell"
    if k == 0:
        return "apex cell"
    return f"{k}-D aggregate cell"

# Лекцийн Жишээ 3.9-ийн 5 cell-ийг ангил
ex = [("Jan", "*", "*"), ("*", "Chicago", "*"), ("Jan", "*", "Business"), ("Jan", "Chicago", "Business"), ("*", "*", "*")]
for c in ex:
    print(c, "->", classify(c))

def is_ancestor(a, b):
    """a нь b-ийн ancestor эсэх: a != b ба a-ийн утга бүр "*" эсвэл b-ийнхтэй тэнцүү."""
    return a != b and all(x == "*" or x == y for x, y in zip(a, b))

# (УБ, Q1, Кофе)-ийн ancestor-ууд, (УБ, *, *)-ийн descendant-уудыг ол
all_cells = [tuple(r) for r in cells[dims].itertuples(index=False)]
anc = [c for c in all_cells if is_ancestor(c, ("УБ", "Q1", "Кофе"))]
desc = [c for c in all_cells if is_ancestor(("УБ", "*", "*"), c)]
print("\n(УБ, Q1, Кофе)-ийн ancestor:", len(anc))
for c in anc:
    print("  ", c)
print("(УБ, *, *)-ийн descendant:", len(desc))
for c in desc:
    print("  ", c)


cell: 27 | томьёо: 27 | True
('Jan', '*', '*') -> 1-D aggregate cell
('*', 'Chicago', '*') -> 1-D aggregate cell
('Jan', '*', 'Business') -> 2-D aggregate cell
('Jan', 'Chicago', 'Business') -> base cell
('*', '*', '*') -> apex cell

(УБ, Q1, Кофе)-ийн ancestor: 7
   ('УБ', 'Q1', '*')
   ('УБ', '*', 'Кофе')
   ('*', 'Q1', 'Кофе')
   ('УБ', '*', '*')
   ('*', 'Q1', '*')
   ('*', '*', 'Кофе')
   ('*', '*', '*')
(УБ, *, *)-ийн descendant: 8
   ('УБ', 'Q1', 'Кофе')
   ('УБ', 'Q1', 'Цай')
   ('УБ', 'Q2', 'Кофе')
   ('УБ', 'Q2', 'Цай')
   ('УБ', 'Q1', '*')
   ('УБ', 'Q2', '*')
   ('УБ', '*', 'Кофе')
   ('УБ', '*', 'Цай')


## 3) Cuboid-ийн lattice ба тоо

Dimension-ы дэд олонлог бүрээр group-by хийвэл нэг cuboid үүснэ. Эдгээрийн lattice нь data cube.

- Hierarchy-гүй n dimension: **2ⁿ** cuboid
- Hierarchy-тай: **T = ∏ (Lᵢ + 1)**, Lᵢ нь i-р dimension-ийн түвшний тоо, `+1` нь `all`

### TASK 3: Lattice ба cuboid-ийн тоо

1. 4 dimension (`time`, `item`, `location`, `supplier`)-ийн lattice-ийг 0-D-ээс 4-D хүртэл түвшин бүрээр тоол.
2. Лекцийн жишээ: `time` (day < month < year, L = 3), `location` (city < country, L = 2), `item` (type, L = 1). Нийт cuboid хэд вэ?
3. 10 dimension, тус бүр 5 түвшинтэй бол хэд вэ?
4. Манай схем: `time` (day, month, year), `location` (district, city, country), `item` (item, type), `customer_group` (1 түвшин). Нийт cuboid хэд вэ?
5. Яагаад бүгдийг урьдчилан тооцоолох (full materialization) практикт боломжгүй вэ? (үгээр)

In [4]:
dims4 = ["time", "item", "location", "supplier"]

# 1. Lattice-ийн түвшин бүрийн cuboid-ийн тоо (0-D ... 4-D) ба нийт
for k in range(len(dims4) + 1):
    cs = list(combinations(dims4, k))
    print(f"{k}-D ({len(cs)}):", [", ".join(c) if c else "apex" for c in cs])
print("нийт:", 2 ** len(dims4))

# 2. Лекцийн жишээ: hier = {"time": 3, "location": 2, "item": 1}; нийт cuboid = prod(L + 1)
hier = {"time": 3, "location": 2, "item": 1}
print("лекцийн жишээ:", math.prod(L + 1 for L in hier.values()))

# 3. 10 dimension x 5 түвшин
print("10 dimension x 5 түвшин:", (5 + 1) ** 10)

# 4. Манай схем: {"time": 3, "location": 3, "item": 2, "customer_group": 1}
ours = {"time": 3, "location": 3, "item": 2, "customer_group": 1}
print("манай схем:", math.prod(L + 1 for L in ours.values()))


0-D (1): ['apex']
1-D (4): ['time', 'item', 'location', 'supplier']
2-D (6): ['time, item', 'time, location', 'time, supplier', 'item, location', 'item, supplier', 'location, supplier']
3-D (4): ['time, item, location', 'time, item, supplier', 'time, location, supplier', 'item, location, supplier']
4-D (1): ['time, item, location, supplier']
нийт: 16
лекцийн жишээ: 24
10 dimension x 5 түвшин: 60466176
манай схем: 96


**Хариулт (5).** 10 dimension, 5 түвшинтэй үед 60 сая гаруй cuboid болно. Бүгдийг тооцоход хэт их зай, хугацаа орох ба өгөгдөл шинэчлэгдэх бүрт дахин бодох хэрэгтэй тул боломжгүй.

## 4) GROUP BY CUBE

SQL-ийн `GROUP BY CUBE (month, city)` нь 2² = 4 cuboid-ийг нэг дор гаргана: `(month, city)`, `(month, NULL)`, `(NULL, city)`, `(NULL, NULL)`. SQL дээр `*` (all)-ийн оронд **NULL** гарна.

### TASK 4: CUBE-ийг өөрөө хэрэгжүүлэх

1. `cube_groupby` функц бич: dimension-ы бүх дэд олонлог тус бүрээр бүлэглэж, нэгтгэгдсэн dimension-д `NaN` (NULL) тавь.
2. `df` дээр `["month", "city"]`-ээр CUBE хий. Cuboid тус бүрд хэдэн мөр гарсныг тоол.
3. Apex мөр (бүх dimension NULL) нийт борлуулалттай тэнцэх ёстой. Шалга.
4. SQLite нь `CUBE`-г дэмждэггүй. `UNION ALL`-аар ижил үр дүнг SQL-ээр гаргаж, pandas-ийн үр дүнтэй харьцуул.

In [5]:
def cube_groupby(df, dims, measure, agg="sum"):
    """GROUP BY CUBE: dimension-ы бүх дэд олонлог тус бүрээр бүлэглэ.
    Нэгтгэгдсэн dimension-д np.nan (NULL) тавь. Баганууд: dims + [measure]."""
    parts = []
    for k in range(len(dims), -1, -1):
        for sub in combinations(dims, k):
            if sub:
                g = df.groupby(list(sub), as_index=False)[measure].agg(agg)
            else:
                g = pd.DataFrame({measure: [df[measure].agg(agg)]})
            for d in dims:
                if d not in sub:
                    g[d] = np.nan
            parts.append(g[dims + [measure]])
    return pd.concat(parts, ignore_index=True)

# 1. ["month", "city"]-ээр CUBE хий; cuboid тус бүрийн мөрийн тоог гарга
cb = cube_groupby(df, ["month", "city"], "sales")
cb["cuboid"] = (cb["month"].notna().map({True: "month", False: "NULL"}) + ", "
                + cb["city"].notna().map({True: "city", False: "NULL"}))
print(cb["cuboid"].value_counts(sort=False))
print("нийт мөр:", len(cb))

# 2. Apex мөр нийт борлуулалттай тэнцэх эсэхийг шалга
apex_val = cb.loc[cb["month"].isna() & cb["city"].isna(), "sales"].iloc[0]
print("apex:", apex_val, "| нийт:", df["sales"].sum(), "|", apex_val == df["sales"].sum())

# 3. SQLite дээр ижил үр дүнг UNION ALL-аар гарга, pandas-ийн үр дүнтэй харьцуул
con = sqlite3.connect(":memory:")
df[["month", "city", "sales"]].to_sql("sales_table", con, index=False)
q = """
SELECT month, city, SUM(sales) AS sales FROM sales_table GROUP BY month, city
UNION ALL
SELECT month, NULL, SUM(sales) FROM sales_table GROUP BY month
UNION ALL
SELECT NULL, city, SUM(sales) FROM sales_table GROUP BY city
UNION ALL
SELECT NULL, NULL, SUM(sales) FROM sales_table
"""
sql = pd.read_sql(q, con)
a = cb[["month", "city", "sales"]].fillna("ALL").sort_values(["month", "city"]).reset_index(drop=True)
b = sql.fillna("ALL").sort_values(["month", "city"]).reset_index(drop=True)
print("SQL мөр:", len(sql), "| ижил:", a.astype(str).equals(b.astype(str)))


cuboid
month, city    36
month, NULL     6
NULL, city      6
NULL, NULL      1
Name: count, dtype: int64
нийт мөр: 49
apex: 2602025 | нийт: 2602025 | True
SQL мөр: 49 | ижил: True


## 5) Concept hierarchy: roll-up ба drill-down

Concept hierarchy нь доод түвшний ойлголтыг дээд түвшний ойлголтод холбох дараалал. Roll-up бол модоор дээш (дүүрэг -> хот -> улс), drill-down бол доош.

| Төрөл | Тайлбар | Жишээ |
|---|---|---|
| Schema hierarchy | Атрибутын бүрэн эсвэл хэсэгчилсэн дараалал | `day < month < quarter < year`; `day < week` |
| Set grouping hierarchy | Утгыг мужаар бүлэглэнэ | `price`: `($0..$200]`, `($200..$400]` |

Интервалд `(` нь эхлэх утгыг оруулахгүй, `]` нь төгсгөлийн утгыг оруулна.

### TASK 5: Шатлал бүрээр нэгтгэх

1. **Байршил**: борлуулалтыг `country`, дараа нь `city`, дараа нь `district`-ээр гарга. Улс бүрийн нийлбэр хот бүрийнхийн нийлбэртэй тэнцэх ёстой.
2. **Drill-down**: `Монгол` улсаас эхлэж хот руу, хамгийн их хотоос дүүрэг рүү задал.
3. **Хугацаа**: `quarter` (Q1, Q2) баганыг нэм. Улирлаар нэгтгэ.
4. **Хэсэгчилсэн дараалал**: `week` нь `month`-ийн дотор бүрэн багтдаггүй. Нэгээс олон сард хамаарсан ISO долоо хоногийг тоол.
5. **Set grouping**: барааны `unit_price`-ийг 200 нэгжийн интервалаар `pd.cut`-аар бүлэглэ (баруун тал хаалттай). Зөвхөн хэдэн барааны үнэ 200 дээр яг тохиолдож, ямар интервалд орохыг шалга. Дараа нь 100 нэгжийн интервалаар нарийвчил (drill-down).

In [6]:
# 1. Байршил: country, city, district-ээр борлуулалтыг гарга; нийлбэрүүд тэнцэх эсэхийг шалга
by_country = df.groupby("country")["sales"].sum()
by_city = df.groupby(["country", "city"])["sales"].sum()
by_district = df.groupby(["country", "city", "district"])["sales"].sum()
print(by_country, "\n")
print(by_city, "\n")
print("country = Σcity:", (by_city.groupby(level=0).sum() == by_country).all(),
      "| city = Σdistrict:", (by_district.groupby(level=[0, 1]).sum() == by_city).all())

# 2. Drill-down: хамгийн их хотоос дүүрэг рүү задал
mn = df[df["country"] == "Монгол"]
city_mn = mn.groupby("city")["sales"].sum().sort_values(ascending=False)
top_city = city_mn.index[0]
print("\nМонгол -> хот:\n", city_mn)
print(f"\n{top_city} -> дүүрэг:\n", mn[mn["city"] == top_city].groupby("district")["sales"].sum()
      .sort_values(ascending=False))

# 3. df["quarter"] баганыг нэм ("Q1", "Q2") ба улирлаар нэгтгэ
df["quarter"] = "Q" + df["date"].dt.quarter.astype(str)
print("\n", df.groupby("quarter")["sales"].sum())

# 4. ISO долоо хоног (df["date"].dt.isocalendar().week) нэгээс олон сард хамаарч байгаа эсэхийг тоол
df["week"] = df["date"].dt.isocalendar().week
wk = df.groupby("week")["month"].nunique()
print("\nнийт долоо хоног:", len(wk), "| 2 сарт хамаарсан:", (wk > 1).sum(), "->", wk[wk > 1].index.tolist())

# 5. Set grouping: dim_item["unit_price"]-ийг 200, дараа нь 100 нэгжийн интервалаар pd.cut-аар бүлэглэ
#    Үнэ нь яг 200 байх барааг ямар интервалд орж байгааг шалга (right=True)
#    Бүлэг тус бүрийн барааны тоо, борлуулалтыг гарга
mx = dim_item["unit_price"].max()
for step in (200, 100):
    bins = np.arange(0, mx + step, step)
    dim_item[f"p{step}"] = pd.cut(dim_item["unit_price"], bins=bins, right=True)
    df[f"p{step}"] = pd.cut(df["unit_price"], bins=bins, right=True)
print("\n", dim_item[["item", "unit_price", "p200", "p100"]])
print("\nүнэ = 200:", dim_item.loc[dim_item["unit_price"] == 200, ["item", "p200"]].values.tolist())
for step in (200, 100):
    print(df.groupby(f"p{step}", observed=True).agg(items=("item", "nunique"), sales=("sales", "sum")), "\n")


country
Монгол    1887750
Хятад      714275
Name: sales, dtype: int64 

country  city       
Монгол   Дархан          333200
         Улаанбаатар    1131800
         Ховд            149875
         Эрдэнэт         272875
Хятад    Бээжин          472975
         Шанхай          241300
Name: sales, dtype: int64 

country = Σcity: True | city = Σdistrict: True

Монгол -> хот:
 city
Улаанбаатар    1131800
Дархан          333200
Эрдэнэт         272875
Ховд            149875
Name: sales, dtype: int64

Улаанбаатар -> дүүрэг:
 district
Сүхбаатар    429450
Хан-Уул      393525
Баянзүрх     308825
Name: sales, dtype: int64

 quarter
Q1    1283550
Q2    1318475
Name: sales, dtype: int64

нийт долоо хоног: 27 | 2 сарт хамаарсан: 4 -> [5, 9, 14, 18]

       item  unit_price        p200        p100
0  Ноутбук         800  (600, 800]  (700, 800]
1  Монитор         300  (200, 400]  (200, 300]
2  Принтер         200    (0, 200]  (100, 200]
3     Утас         400  (200, 400]  (300, 400]
4   Таблет       

## 6) Measure: distributive, algebraic, holistic

| Төрөл | Жишээ | Хэсгийн үр дүнгээс нийт тооцох |
|---|---|---|
| Distributive | `sum`, `count`, `min`, `max` | Хэсгийн үр дүнг нэгтгэхэд бүтнээр нь бодсонтой адил |
| Algebraic | `avg`, `stddev`, `min_N`, `max_N` | Тогтмол M ширхэг distributive функцээс |
| Holistic | `median`, `mode`, `rank` | Боломжгүй, бүх утгыг хадгална |

### TASK 6: Хэмжүүрийн нэгтгэлийн шалгалт

1. Лекцийн жишээ: Салбар A = 10, 20, 30; Салбар B = 40, 50. `sum`, `count`, `avg`, `median`-ийг A, B, бүгд гэж тус бүр бод. Хэсгээс нэгтгэж болох, болохгүйг тогтоо.
2. Бодит өгөгдөл: барааны `type` бүрийг нэг хэсэг гэж үзэж `sales`-ийн **дундаж**-ыг (a) бүх мөрөөр, (b) `type` бүрийн дундажуудын дундажаар, (c) `type` бүрийн `(sum, count)` хосоор бод. Аль нь зөв бэ? Яагаад (b) буруу байна вэ?
3. **Стандарт хазайлт**-ыг `type` бүрийн `(n, Σx, Σx²)` хосоос бод. Бүтнээр нь бодсонтой тэнцэх ёстой (`ddof = 0`).
4. **median**-ыг `type`-ийн median-уудын median-аар бодоод бүтнийхтэй харьцуул.
5. **max_N** (top 3): `type` бүрийн top 3-ыг нэгтгэж дахин top 3 авбал бүтэн өгөгдлийн top 3-тай тэнцэх үү?

In [7]:
# 1. Лекцийн жишээ: A = [10, 20, 30], B = [40, 50]
#    sum, count, avg, median-ийг A, B, бүгд гэж бод; хэсгээс нэгтгэж болохыг шалга
A, B = [10, 20, 30], [40, 50]
AB = A + B
for name, f in [("sum", np.sum), ("count", len), ("avg", np.mean), ("median", np.median)]:
    print(f"{name:6}  A={f(A):<6} B={f(B):<6} бүгд={f(AB)}")
print("sum: ", sum(A) + sum(B), "| count:", len(A) + len(B))
print("avg-ийн avg:", (np.mean(A) + np.mean(B)) / 2, "| (sum, count)-оор:", (sum(A) + sum(B)) / (len(A) + len(B)))
print("median-ийн median:", np.median([np.median(A), np.median(B)]), "| жинхэнэ:", np.median(AB))

# 2. Бодит өгөгдөл: df.groupby("type")["sales"] -ийг хэсгүүд гэж үз
#    (a) бүх мөрийн дундаж, (b) type-ийн дундажуудын дундаж, (c) (sum, count) хосоор бодсон дундаж
g = df.groupby("type")["sales"]
s = g.agg(["sum", "count", "mean"])
print("\n", s)
a = df["sales"].mean()
b = s["mean"].mean()
c = s["sum"].sum() / s["count"].sum()
print(f"(a) {a:.4f}  (b) {b:.4f}  (c) {c:.4f}")

# 3. Стандарт хазайлтыг type бүрийн (n, sum x, sum x^2)-аас бод; df["sales"].std(ddof=0)-тэй харьцуул
p = g.agg(n="count", s1="sum", s2=lambda x: (x ** 2).sum())
N, S1, S2 = p["n"].sum(), p["s1"].sum(), p["s2"].sum()
std_parts = math.sqrt(S2 / N - (S1 / N) ** 2)
print(f"\nstd хэсгээс: {std_parts:.4f} | бүтнээр: {df['sales'].std(ddof=0):.4f}",
      np.isclose(std_parts, df["sales"].std(ddof=0)))

# 4. median-ыг type-ийн median-уудын median-тай харьцуул
print("\n", g.median())
print("median-ийн median:", g.median().median(), "| бүтнээр:", df["sales"].median())

# 5. max_N (top 3): type бүрийн top 3-ыг нэгтгэж дахин top 3 ав; бүтнийхтэй харьцуул
top_parts = g.nlargest(3).sort_values(ascending=False).head(3).values
top_all = df["sales"].nlargest(3).values
print("\ntop3 хэсгээс:", top_parts, "| бүтнээр:", top_all, "|", (top_parts == top_all).all())


sum     A=60     B=90     бүгд=150
count   A=3      B=2      бүгд=5
avg     A=20.0   B=45.0   бүгд=30.0
median  A=20.0   B=45.0   бүгд=30.0
sum:  150 | count: 5
avg-ийн avg: 32.5 | (sum, count)-оор: 30.0
median-ийн median: 32.5 | жинхэнэ: 30.0

                         sum  count         mean
type                                           
Гар утас ба таблет  1172400    879  1333.788396
Дагалдах хэрэгсэл    221125   1144   193.291084
Компьютер           1208500    977  1236.949846
(a) 867.3417  (b) 921.3431  (c) 867.3417

std хэсгээс: 858.2343 | бүтнээр: 858.2343 True

 type
Гар утас ба таблет    1200.0
Дагалдах хэрэгсэл      150.0
Компьютер              900.0
Name: sales, dtype: float64
median-ийн median: 900.0 | бүтнээр: 600.0

top3 хэсгээс: [4000 4000 4000] | бүтнээр: [4000 4000 4000] | True


**Хариулт.** (a), (c) хоёулаа 867.34 буюу зөв. (b) 921.34 буруу, учир нь type бүрийн дундаж ижил жин авч байна, гэтэл бүлгүүдийн мөрийн тоо өөр. std (n, Σx, Σx²)-аас зөв гарсан. median-ийн median 900, жинхэнэ нь 600 тул median-ыг хэсгээс бодож болохгүй. top 3 зөв гарсан.

## 7) Query-д cuboid сонгох

Query ирэхэд хамгийн бага хэмжээтэй, асуултыг хариулж чадах materialized cuboid-ийг сонгоно.

1. Query-ийн GROUP BY ба WHERE-ээс шаардлагатай түвшнийг тогтооно.
2. Hierarchy-гээр тухайн түвшин эсвэл **илүү нарийвчилсан** түвшний cuboid-ийг хайна (хойно roll-up хийж болно).
3. Хамгийн бага өртөгтэй (мөрийн тоо хамгийн цөөн) cuboid-ийг сонгоно.

Cuboid-ийг `(time түвшин, location түвшин, item түвшин)` гэж тэмдэглэнэ, `None` нь `all`. Жишээ нь `("month", "city", None)`.

### TASK 7: Cuboid сонгогч

Дараах cuboid-ууд materialized: `("day","district","item")`, `("month","city","item")`, `("month","city",None)`, `("month","country","item")`, `("year","country",None)`.

1. `can_answer`: materialized cuboid нь query-г хариулж чадах нөхцөл. Dimension бүрт materialized түвшин query-ийн түвшингээс **нарийн эсвэл тэнцүү** байх ёстой.
2. `size`: cuboid-ийн мөрийн тоо (өгөгдлөөс).
3. Дараах query-үүдэд хариулж чадах cuboid-уудыг жагсааж, хамгийн бага хэмжээтэйг сонго:
   - Q1: `("month", "country", None)`
   - Q2: `("year", "city", "type")`
   - Q3: `(None, None, None)` (apex)
4. Q2-ыг `("month","city",None)` cuboid яагаад хариулж чадахгүй вэ? (үгээр)

In [8]:
df["day"]  = df["date"].dt.strftime("%Y-%m-%d")
df["year"] = df["date"].dt.year

LEVELS = {"time": ["day", "month", "year"],
          "location": ["district", "city", "country"],
          "item": ["item", "type"]}
DIMS = ["time", "location", "item"]

def rank(dim, level):
    return LEVELS[dim].index(level) if level else len(LEVELS[dim])    # None = all, хамгийн ерөнхий

def can_answer(mat, q):
    """materialized cuboid mat нь query q-г хариулж чадах уу?
    Dimension бүрт mat-ийн түвшин q-ийн түвшингээс нарийн эсвэл тэнцүү байх ёстой."""
    return all(rank(d, m) <= rank(d, l) for d, m, l in zip(DIMS, mat, q))

def size(c):
    """Cuboid-ийн мөрийн тоо (өгөгдлөөс). None-ийг алгас, бүгд None бол 1."""
    cols = [l for l in c if l is not None]
    return len(df[cols].drop_duplicates()) if cols else 1

materialized = [("day", "district", "item"), ("month", "city", "item"), ("month", "city", None),
                ("month", "country", "item"), ("year", "country", None)]
queries = [("month", "country", None), ("year", "city", "type"), (None, None, None)]

# Query бүрт хариулж чадах cuboid-уудыг жагсааж, хамгийн бага хэмжээтэйг сонго
for m in materialized:
    print(m, size(m))
for q in queries:
    ok = [(m, size(m)) for m in materialized if can_answer(m, q)]
    best = min(ok, key=lambda x: x[1])
    print("\nquery", q)
    for m, sz in ok:
        print("   ", m, sz)
    print("  -> сонгох:", best[0], best[1])


('day', 'district', 'item') 2616
('month', 'city', 'item') 288
('month', 'city', None) 36
('month', 'country', 'item') 96
('year', 'country', None) 2

query ('month', 'country', None)
    ('day', 'district', 'item') 2616
    ('month', 'city', 'item') 288
    ('month', 'city', None) 36
    ('month', 'country', 'item') 96
  -> сонгох: ('month', 'city', None) 36

query ('year', 'city', 'type')
    ('day', 'district', 'item') 2616
    ('month', 'city', 'item') 288
  -> сонгох: ('month', 'city', 'item') 288

query (None, None, None)
    ('day', 'district', 'item') 2616
    ('month', 'city', 'item') 288
    ('month', 'city', None) 36
    ('month', 'country', 'item') 96
    ('year', 'country', None) 2
  -> сонгох: ('year', 'country', None) 2


**Хариулт (4).** `("month","city",None)` cuboid-д item аль хэдийн all болсон тул type-аар задлах боломжгүй.

## 8) Materialization: Full, No, Partial

| Арга | Тодорхойлолт | Онцлог |
|---|---|---|
| Full | Бүх cuboid-ийг урьдчилан тооцно | Query хурдан, хадгалалт их |
| No | Нэгтгэлийг хэрэгтэй үед тооцно | Хадгалалт бага, query удаан |
| Partial | Түгээмэл cuboid-уудыг хадгална | Хурд ба хадгалалтын тэнцвэр |

### TASK 8: 3 dimension-ийн cube (item, city, month)

1. 8 cuboid-ийн мөрийн тоог гарга. **Full** materialization-ий нийт хадгалалт, **No** materialization-ий хадгалалт (зөвхөн base cuboid) хэд вэ?
2. Query бүр өөрийн cuboid-ийг хариулахад, хариулж чадах хамгийн жижиг materialized cuboid-ийг уншина гэж үзвэл 8 cuboid бүрийн query-ийн **дундаж өртөг** хэд вэ? (Full ба No-д тус тусад нь.)
3. **Partial**: base-ээс гадна 2 cuboid сонгож (нийт 3), дундаж өртгийг хамгийн бага болгох жагсаалтыг greedy аргаар ол.
4. Partial-ийн хадгалалт ба дундаж өртгийг Full, No-той харьцуул. Лекцийн "зөвхөн 3 cuboid хадгалбал ямар materialization вэ?" асуултад хариул.

In [9]:
D3 = ["item", "city", "month"]

# 1. 8 cuboid-ийн мөрийн тоо (sizes). Full ба No materialization-ий хадгалалт
cuboids = [c for k in range(len(D3), -1, -1) for c in combinations(D3, k)]
sizes = {c: (len(df[list(c)].drop_duplicates()) if c else 1) for c in cuboids}
for c in cuboids:
    print(c if c else "apex", sizes[c])
base = tuple(D3)
print("Full хадгалалт:", sum(sizes.values()), "| No хадгалалт:", sizes[base])

# 2. cost(q, mat): query q-г хариулах хамгийн жижиг materialized cuboid-ийн хэмжээ
#    avg_cost(mat): 8 cuboid-ийн query-ийн дундаж өртөг. Full ба No-д бод
def cost(q, mat):
    return min(sizes[m] for m in mat if set(q) <= set(m))

def avg_cost(mat):
    return np.mean([cost(q, mat) for q in cuboids])

print("Full дундаж өртөг:", avg_cost(cuboids), "| No дундаж өртөг:", avg_cost([base]))

# 3. Greedy: base-ээс гадна 2 cuboid сонго (нийт 3), сонгох бүрт дундаж өртөг хамгийн их буурахыг
mat = [base]
for _ in range(2):
    best = min((c for c in cuboids if c not in mat), key=lambda c: avg_cost(mat + [c]))
    mat.append(best)
    print("нэмэв:", best, "-> дундаж өртөг", avg_cost(mat))

# 4. Partial-ийн хадгалалт, дундаж өртгийг Full, No-той харьцуул
summary = pd.DataFrame({
    "хадгалалт": [sum(sizes.values()), sizes[base], sum(sizes[m] for m in mat)],
    "дундаж өртөг": [avg_cost(cuboids), avg_cost([base]), avg_cost(mat)]},
    index=["Full", "No", "Partial"])
print(summary)


('item', 'city', 'month') 288
('item', 'city') 48
('item', 'month') 48
('city', 'month') 36
('item',) 8
('city',) 6
('month',) 6
apex 1
Full хадгалалт: 441 | No хадгалалт: 288
Full дундаж өртөг: 55.125 | No дундаж өртөг: 288.0
нэмэв: ('city', 'month') -> дундаж өртөг 162.0
нэмэв: ('item', 'city') -> дундаж өртөг 102.0
         хадгалалт  дундаж өртөг
Full           441        55.125
No             288       288.000
Partial        372       102.000


**Хариулт (4).** Full: 441 мөр, дундаж өртөг 55.1. No: 288 мөр, өртөг 288. Partial (base, (city, month), (item, city)): 372 мөр, өртөг 102. Зөвхөн 3 cuboid хадгалж байгаа тул **partial materialization**.

## 9) Iceberg cube, antimonotonicity ба BUC

**Iceberg cube**: `COUNT(*) >= min_sup` нөхцөл хангасан cell-үүдийг л хадгалсан cube.

```
compute cube sales_iceberg as
select month, city, customer_group, count(*)
from salesInfo cube by month, city, customer_group
having count(*) >= min_sup
```

**Antimonotonicity**: нэг cell босго хангахгүй бол түүнээс нарийвчилсан бүх descendant ч хангахгүй. **BUC** (Bottom-Up Computation) нь ингэж **pruning** хийж iceberg cube-ийг үр ашигтай тооцдог.

### TASK 9: Лекцийн дасгал (min_sup = 3) ба BUC

`cs_iceberg_demo.csv` нь 10 бичлэгтэй (item, city): Ном УБ x4, Ном Дархан x1, Дэвтэр УБ x3, Дэвтэр Дархан x2.

1. `(item, city)`-ийн COUNT-ийг гарга. `min_sup = 3` үед Iceberg cube-д аль cell-үүд үлдэх вэ? (CUBE-ийн бүх cuboid-ийг оруул.)
2. `buc` функцийг бөглөж ажиллуул. Аль cell-үүдийн цаашдын салбарыг **prune** хийсэн бэ?
3. Том өгөгдөл: `["month", "city", "customer_group", "item"]`-ээр `min_sup = 30` үед BUC-ийн үр дүн CUBE-ээр бүгдийг тооцоод шүүсэнтэй ижил эсэхийг шалга. Хэдэн cell үлдэв, хэдэн partition хянагдав, нийт cell хэд байв?
4. Antimonotonicity-г шалга: iceberg cell бүрийн ancestor бүр мөн iceberg-д орсон эсэх.

In [10]:
demo = pd.read_csv("cs_iceberg_demo.csv")
min_sup = 3

# 1. (item, city)-ийн COUNT; cube_groupby-аар бүх cuboid-ийг гаргаж min_sup-аар шүү
demo["cnt"] = 1
print(demo.groupby(["item", "city"]).size())
dc = cube_groupby(demo, ["item", "city"], "cnt")
print(dc.fillna("*"))
print("\nIceberg (count >= 3):")
print(dc[dc["cnt"] >= min_sup].fillna("*"))

def buc(df, dims, min_sup):
    """BUC: (cell, count) жагсаалт, prune хийсэн cell-ийн жагсаалт, хянасан partition-ийн тоог буцаа.
    Санамж: рекурсив rec(part, prefix, start). Partition бүрийн COUNT >= min_sup бол
    цааш үргэлжлүүл, үгүй бол prune."""
    out, pruned = [], []
    n_part = 0

    def rec(part, prefix, start):
        nonlocal n_part
        out.append((prefix, len(part)))
        for i in range(start, len(dims)):
            for v, sub in part.groupby(dims[i], sort=True):
                n_part += 1
                cell = prefix[:i] + (v,) + prefix[i + 1:]
                if len(sub) >= min_sup:
                    rec(sub, cell, i + 1)
                else:
                    pruned.append((cell, len(sub)))

    if len(df) >= min_sup:
        rec(df, ("*",) * len(dims), 0)
    return out, pruned, n_part

# 2. buc(demo, ["item", "city"], 3): аль cell-үүдийг prune хийв?
res, pr, n = buc(demo, ["item", "city"], 3)
print("\nBUC үр дүн:")
for c, k in res:
    print("  ", c, k)
print("prune:", pr, "| partition:", n)

# 3. d4 = ["month", "city", "customer_group", "item"], min_sup = 30
#    BUC-ийн үр дүн CUBE-ээс шүүсэнтэй ижил эсэх; үлдсэн cell, хянасан partition, нийт cell
d4 = ["month", "city", "customer_group", "item"]
res4, pr4, n4 = buc(df, d4, 30)
df["cnt"] = 1
full = cube_groupby(df, d4, "cnt")
full_ice = full[full["cnt"] >= 30].fillna("*")
set_cube = {tuple(r) for r in full_ice[d4 + ["cnt"]].itertuples(index=False)}
set_buc = {c + (k,) for c, k in res4}
print("\nижил:", set_cube == set_buc, "| iceberg cell:", len(set_buc),
      "| хянасан partition:", n4, "| prune:", len(pr4), "| нийт cell:", len(full))

# 4. Antimonotonicity: iceberg cell бүрийн ancestor бүр мөн iceberg-д орсон эсэх
ice_cells = {c for c, k in res4}

def ancestors(c):
    idx = [i for i, v in enumerate(c) if v != "*"]
    for r in range(len(idx)):
        for keep in combinations(idx, r):
            yield tuple(c[i] if i in keep else "*" for i in range(len(c)))

print("antimonotone:", all(a in ice_cells for c in ice_cells for a in ancestors(c)))


item    city  
Дэвтэр  Дархан    2
        УБ        3
Ном     Дархан    1
        УБ        4
dtype: int64


     item    city  cnt
0  Дэвтэр  Дархан    2
1  Дэвтэр      УБ    3
2     Ном  Дархан    1
3     Ном      УБ    4
4  Дэвтэр       *    5
5     Ном       *    5
6       *  Дархан    3
7       *      УБ    7
8       *       *   10

Iceberg (count >= 3):
     item    city  cnt
1  Дэвтэр      УБ    3
3     Ном      УБ    4
4  Дэвтэр       *    5
5     Ном       *    5
6       *  Дархан    3
7       *      УБ    7
8       *       *   10

BUC үр дүн:
   ('*', '*') 10
   ('Дэвтэр', '*') 5
   ('Дэвтэр', 'УБ') 3
   ('Ном', '*') 5
   ('Ном', 'УБ') 4
   ('*', 'Дархан') 3
   ('*', 'УБ') 7
prune: [(('Дэвтэр', 'Дархан'), 2), (('Ном', 'Дархан'), 1)] | partition: 8



ижил:

 True | iceberg cell: 310 | хянасан partition: 1135 | prune: 826 | нийт cell: 1636
antimonotone: True


## 10) Closed cell, quotient cube, shell fragment

- **Closed cell**: өөрөөсөө илүү нарийвчилсан, **яг ижил суурь нүднүүдийг (base cells) хамарсан** өөр cell байхгүй cell.
- **Quotient cube**: зөвхөн closed cell-үүдээс бүрдсэн шахсан cube.
- **Shell fragment**: dimension-уудыг жижиг бүлэгт хувааж зөвхөн тэдгээрийн cuboid-ийг урьдчилан тооцох арга.

### TASK 10: Closed cell олох

`cs_closed_demo.csv` нь 5 суурь мөртэй: Дархан Q1 Кофе, Дархан Q1 Цай, УБ Q2 Кофе, УБ Q2 Цай, УБ Q3 Цай.

1. `cube_cells`-ээр (TASK 2) бүх non-empty cell-ийг гарга. Нийт хэдэн cell байна вэ?
2. Cell бүрийн **cover** (хамарсан суурь мөрийн индексийн олонлог)-ийг тооц.
3. Closed cell нь: өөрийн cover-тэй **яг ижил cover-тэй descendant байхгүй** cell. Closed cell-үүдийг ол.
4. `(Дархан, *, *)` closed уу? Яагаад? Түүнийг төлөөлөх closed cell аль вэ?
5. Quotient cube-д хэдэн cell үлдэх вэ? Шахалтын харьцаа хэд вэ?
6. Shell fragment: 4 dimension-ийг `{time, item}` ба `{location, supplier}` хоёр бүлэгт хувааж, бүлэг доторх cuboid (apex-гүй)-ийг л тооцвол хэдэн cuboid тооцох вэ? Бүтэн cube-тэй (apex-гүй 15) харьцуул.

In [11]:
cd = pd.read_csv("cs_closed_demo.csv")
dims_c = ["branch", "quarter", "item"]

# 1. cube_cells (TASK 2)-ээр бүх non-empty cell-ийг гарга
cc = cube_cells(cd, dims_c, "sales")
cell_list = [tuple(r) for r in cc[dims_c].itertuples(index=False)]
print("non-empty cell:", len(cell_list))

# 2. covers(cell): cell-ийн хамарсан суурь мөрийн индексийн олонлог (frozenset)
def covers(cell):
    m = np.ones(len(cd), dtype=bool)
    for d, v in zip(dims_c, cell):
        if v != "*":
            m &= (cd[d] == v).values
    return frozenset(int(i) for i in np.flatnonzero(m))

cov = {c: covers(c) for c in cell_list}

# 3. is_closed(cell): өөрийнхтэй яг ижил cover-тэй descendant байхгүй (is_ancestor ашигла)
def is_closed(cell):
    return not any(is_ancestor(cell, o) and cov[o] == cov[cell] for o in cell_list)

# 4. Closed cell-ийн жагсаалт; ("Дархан", "*", "*") closed эсэх
closed = [c for c in cell_list if is_closed(c)]
print("closed cell:", len(closed))
for c in closed:
    print("  ", c, sorted(cov[c]))
dk = ("Дархан", "*", "*")
print(dk, "closed:", is_closed(dk), "| cover:", sorted(cov[dk]))
print("төлөөлөх closed cell:", [c for c in closed if cov[c] == cov[dk]])

# 5. Quotient cube-ийн cell-ийн тоо, шахалтын харьцаа
print("quotient cube:", len(closed), "/", len(cell_list), "=", round(len(closed) / len(cell_list), 3))

# 6. Shell fragment: {time, item}, {location, supplier} бүлгийн cuboid (apex-гүй)-ийн тоо, бүтэн cube-тэй харьцуул
groups = [["time", "item"], ["location", "supplier"]]
shell = sum(2 ** len(g) - 1 for g in groups)
print("shell fragment:", shell, "| бүтэн cube (apex-гүй):", 2 ** 4 - 1)


non-empty cell: 25
closed cell: 12
   ('Дархан', 'Q1', 'Кофе') [0]
   ('Дархан', 'Q1', 'Цай') [1]
   ('УБ', 'Q2', 'Кофе') [2]
   ('УБ', 'Q2', 'Цай') [3]
   ('УБ', 'Q3', 'Цай') [4]
   ('Дархан', 'Q1', '*') [0, 1]
   ('УБ', 'Q2', '*') [2, 3]
   ('УБ', '*', 'Цай') [3, 4]
   ('УБ', '*', '*') [2, 3, 4]
   ('*', '*', 'Кофе') [0, 2]
   ('*', '*', 'Цай') [1, 3, 4]
   ('*', '*', '*') [0, 1, 2, 3, 4]
('Дархан', '*', '*') closed: False | cover: [0, 1]
төлөөлөх closed cell: [('Дархан', 'Q1', '*')]
quotient cube: 12 / 25 = 0.48
shell fragment: 6 | бүтэн cube (apex-гүй): 15


**Хариулт (4).** `(Дархан, *, *)` closed биш. Дархан-ы бүх мөр Q1-д байгаа тул `(Дархан, Q1, *)` яг ижил {0, 1} мөрийг хамарна. Түүнийг `(Дархан, Q1, *)` төлөөлнө.

## 11) OLAP серверийн архитектур: ROLAP ба MOLAP

| Архитектур | Хадгалалт | Нэгтгэл |
|---|---|---|
| ROLAP | Relational хүснэгт | SQL |
| MOLAP | Олон хэмжээст массив | Массивын индекс, тэнхлэгээр нэгтгэх |
| HOLAP | Хослуулсан | Нэгтгэсэн хэсэг MOLAP, дэлгэрэнгүй хэсэг ROLAP |

### TASK 11: Ижил асуулга, хоёр архитектур

1. **ROLAP**: SQLite-д `fact` хүснэгт үүсгэж `GROUP BY city, month`-ээр SUM гарга.
2. **MOLAP**: `item x city x month` dense numpy массив үүсгэ (`np.add.at`). `item` тэнхлэгийг roll-up хий (`sum(axis=0)`).
3. Хоёр үр дүн ижил эсэхийг шалга.
4. **Sparsity**: `district x item x day x customer_group` dimension-ийн dense массивын нийт cell-ийн тоо, **хоосон биш** cell-ийн тоо, хоосон биш cell-ийн хувийг гарга. Яагаад олон dimension-тэй cube-д MOLAP-ийн dense массив хэмжээ зарцуулагддаг вэ? HOLAP яагаад үүнийг засдаг вэ? (үгээр)

In [12]:
# 1. ROLAP: SQLite-д "fact" хүснэгт үүсгэж, GROUP BY city, month-ээр SUM гарга
con = sqlite3.connect(":memory:")
df[["item", "city", "month", "sales"]].to_sql("fact", con, index=False)
rolap = pd.read_sql("SELECT city, month, SUM(sales) AS sales FROM fact GROUP BY city, month", con)
print(rolap.head(), len(rolap))

# 2. MOLAP: item x city x month dense numpy массив үүсгэ (np.add.at), item тэнхлэгийг sum-аар roll-up хий
items, cities, months = (sorted(df[c].unique()) for c in ["item", "city", "month"])
ii = pd.Categorical(df["item"], categories=items).codes
ci = pd.Categorical(df["city"], categories=cities).codes
mi = pd.Categorical(df["month"], categories=months).codes
arr = np.zeros((len(items), len(cities), len(months)))
np.add.at(arr, (ii, ci, mi), df["sales"].to_numpy())
molap = arr.sum(axis=0)
print("массив:", arr.shape, "| roll-up:", molap.shape)

# 3. Хоёр үр дүн ижил эсэхийг np.allclose-оор шалга
r = (rolap.pivot(index="city", columns="month", values="sales")
          .reindex(index=cities, columns=months).fillna(0).to_numpy())
print("ROLAP == MOLAP:", np.allclose(r, molap))

# 4. Sparsity: district x item x day x customer_group dimension-ийн dense cell-ийн тоо,
#    хоосон биш cell-ийн тоо, хувь
sd = ["district", "item", "day", "customer_group"]
print({d: df[d].nunique() for d in sd})
dense = math.prod(df[d].nunique() for d in sd)
nonempty = df.groupby(sd).ngroups
print("dense:", dense, "| хоосон биш:", nonempty, "|", round(nonempty / dense * 100, 2), "%")


     city    month  sales
0  Бээжин  2026-01  82050
1  Бээжин  2026-02  83275
2  Бээжин  2026-03  62775
3  Бээжин  2026-04  82300
4  Бээжин  2026-05  73800 36
массив: (8, 6, 6) | roll-up: (6, 6)
ROLAP == MOLAP: True
{'district': 9, 'item': 8, 'day': 181, 'customer_group': 3}
dense: 39096 | хоосон биш: 2829 | 7.24 %


**Хариулт (4).** 39096 cell-ээс 2829 (7.24%) нь л утгатай, бусад нь 0. Dimension нэмэгдэх тусам массив хурдан томордог ч өгөгдөл тэр хэмжээгээр нэмэгддэггүй. HOLAP нягт хэсгийг массивт, сийрэг хэсгийг хүснэгтэд хадгална.

## 12) Discovery-driven exploration

Сая сая cell-ээс сонирхолтойг гараар хайх боломжгүй. Хүлээгдэж буй утгаас ихээр зөрсөн cell-ийг стандартчилсан хазайлтаар (жишээ нь |z| > 2.5) автоматаар тэмдэглэнэ.

- **SelfExp**: cell өөрөө хэр онцгой
- **InExp**: cell-ийн доор (drill-down) онцгой утга нуугдсан эсэх
- **PathExp**: drill-down зам бүрийн онцгой байдал

### TASK 12: SelfExp ба InExp

`cs_monthly_item_city.csv`-д 4 бараа x 6 хот x сар (0 = 2025 оны 12-р сар).

1. Барааны сарын борлуулалтын өөрчлөлтийг (%) гарга: `chg` (бараа x сар 1-6). Лекцийн хүснэгттэй адил эсэхийг шалга.
2. **SelfExp**: хүлээгдэх утгыг мөрийн дундаж + баганын дундаж - нийт дундаж гэж үзэж, үлдэгдэл (residual)-ийг стандартчил. Хамгийн онцгой 4 cell, мөн |z| > 2 болох cell-ийн тоог ол (жижиг хүснэгт тул босгыг 2 гэж авна).
3. **InExp**: cell бүрийг хот руу drill-down хийж, хотын өөрчлөлт нь улсын өөрчлөлтөөс (`chg`) хэр зөрөхийг стандартчил. Бараа x сар бүрт хотуудын хамгийн их |z|-ийг ол.
4. SelfExp өндөр боловч InExp нам, мөн SelfExp нам боловч InExp өндөр cell байна уу? Тэдгээрийг тайлбарла. (Лекцийн Принтер 4-р сар жишээ)

In [13]:
mo = pd.read_csv("cs_monthly_item_city.csv")

# 1. nat: бараа x сар (улсын нийлбэр); chg = pct_change(axis=1) * 100, сар 1-6
nat = mo.pivot_table(index="item", columns="month", values="sales", aggfunc="sum")
chg = ((nat / nat.shift(1, axis=1) - 1) * 100).loc[:, 1:6]
print(chg.round(1))

# 2. SelfExp: хүлээгдэх утга = мөрийн дундаж + баганын дундаж - нийт дундаж
#    residual = chg - хүлээгдэх утга; z = residual / residual.std(); хамгийн онцгой 4 cell, |z| > 2 тоо
expv = chg.mean(axis=1).to_numpy()[:, None] + chg.mean(axis=0).to_numpy()[None, :] - chg.to_numpy().mean()
resid = chg - expv
z_self = resid / resid.stack().std()
print("\nSelfExp z:\n", z_self.round(2))
top = z_self.stack().abs().sort_values(ascending=False)
print("\nхамгийн онцгой 4:\n", top.head(4).round(2))
print("|z| > 2:", (top > 2).sum())

# 3. InExp: бараа x хот x сарын өөрчлөлт (cchg); хотын өөрчлөлт - улсын өөрчлөлт (chg); стандартчил
#    Бараа x сар бүрт хотуудын хамгийн их |z|-ийг ол; хамгийн онцгой cell ба тэр хот
cm = mo.pivot_table(index=["item", "city"], columns="month", values="sales", aggfunc="sum")
cchg = ((cm / cm.shift(1, axis=1) - 1) * 100).loc[:, 1:6]
diff = cchg - chg.reindex(cchg.index.get_level_values("item")).to_numpy()
z_in = diff / diff.stack().std()
in_max = z_in.abs().groupby(level="item").max()
print("\nInExp (хотуудын max |z|):\n", in_max.round(2))
zs = z_in.abs().stack()
it, ct, mn_ = zs.idxmax()
print("хамгийн онцгой:", it, ct, int(mn_), "сар |z| =", round(zs.max(), 2))

comp = pd.DataFrame({"SelfExp": z_self.abs().stack(), "InExp": in_max.stack()}).round(2)
print("\n", comp.sort_values("InExp", ascending=False).head(6))


month      1    2     3    4     5    6
item                                   
Ноутбук  2.0  3.0   1.0  4.0   2.0  3.0
Принтер -1.0  0.1   0.9 -2.0   0.1  0.9
Утас     1.0  2.0 -14.0  2.0   1.0  3.0
Чихэвч   2.9  2.2   3.9  1.0  22.0  2.0

SelfExp z:
 month       1     2     3     4     5     6
item                                       
Ноутбук  0.01  0.12  0.55  0.48 -1.18  0.02
Принтер -0.06  0.05  1.17 -0.30 -1.01  0.15
Утас     0.57  0.67 -2.23  0.81 -0.63  0.81
Чихэвч  -0.52 -0.84  0.50 -0.99  2.82 -0.97

хамгийн онцгой 4:
 item     month
Чихэвч   5        2.82
Утас     3        2.23
Ноутбук  5        1.18
Принтер  3        1.17
dtype: float64
|z| > 2: 2

InExp (хотуудын max |z|):
 month       1     2     3      4     5     6
item                                        
Ноутбук  0.50  0.65  0.52   0.28  0.32  0.56
Принтер  0.60  1.25  0.78  10.56  0.33  0.46
Утас     0.56  0.43  0.33   0.61  0.77  0.57
Чихэвч   0.83  1.10  0.49   0.43  0.56  0.60
хамгийн онцгой: Принтер Ховд 4 с

**Хариулт (4).** Чихэвч 5-р сар, Утас 3-р сар SelfExp өндөр ч InExp бага, бүх хот хамт өөрчлөгдсөн. Принтер 4-р сар SelfExp 0.30 ч InExp 10.56, Ховд хотод огцом унасан. Лекцийн Принтер 4-р сарын жишээтэй адил.

## 13) Prediction cube

Cell бүрт тухайн cell-ийн өгөгдлөөр сургасан загварын **accuracy**-г measure болгон хадгална. Хоёр хэлбэр: (1) загварын accuracy, (2) attribute-ийн predictiveness, өөрөөр хэлбэл attribute-тэй ба гүй загварын accuracy-ийн зөрүү.

`cs_loans.csv`: зээлийн шийдвэр (`approved`). Dimension: `city`, `year`. Feature: `age`, `income`, `debt_ratio`, `gender`.

### TASK 13: Зээлийн prediction cube

1. (`city`, `year`) cell бүр дээр `age`, `income`, `debt_ratio`-оор logistic regression-ийг 5-fold cross validation-ээр сургаж accuracy-г гарга (`StandardScaler` ашигла).
2. Cell бүрт `gender`-ийг нэмж accuracy-г дахин гарга. Зөрүү (`gender_gain`) нь gender-ийн predictiveness.
3. Accuracy-ийн (хот x жил) хүснэгтийг гарга. Загвар хамгийн сул хот аль вэ? Яагаад (cell дэх өгөгдлийн хэмжээ, шуугианыг бод)?
4. Аль хотод `gender`-ийг нэмэхэд accuracy тогтвортой өсч байна вэ? Ховд-д зөрүү яагаад тогтворгүй байна вэ? Энэ нь fairness-ийн ямар сэрэмжлүүлэг вэ? (үгээр)

In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

loans = pd.read_csv("cs_loans.csv")

def cell_acc(g, feats):
    """Нэг cell (g)-ийн өгөгдөл дээр feats-ээр logistic regression-ийг
    5-fold cross validation-ээр сургаж дундаж accuracy-г буцаа (StandardScaler-тай pipeline)."""
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    cv = KFold(n_splits=5, shuffle=True, random_state=0)
    return cross_val_score(model, g[feats], g["approved"], cv=cv, scoring="accuracy").mean()

# 1. (city, year) cell бүрт age, income, debt_ratio-оор accuracy
# 2. gender нэмсэн accuracy ба зөрүү (gender_gain)
base_f = ["age", "income", "debt_ratio"]
rows = []
for (city, year), g in loans.groupby(["city", "year"]):
    a0 = cell_acc(g, base_f)
    a1 = cell_acc(g, base_f + ["gender"])
    rows.append({"city": city, "year": year, "n": len(g), "approve_rate": g["approved"].mean(),
                 "acc": a0, "acc_gender": a1, "gender_gain": a1 - a0})

# 3. pc DataFrame үүсгэж, accuracy (хот x жил) ба gender_gain-ийн хүснэгтийг pivot-оор гарга
pc = pd.DataFrame(rows)
print(pc.round(3))
print("\naccuracy:\n", pc.pivot(index="city", columns="year", values="acc").round(3))
print("\ngender_gain:\n", pc.pivot(index="city", columns="year", values="gender_gain").round(3))
print("\nn:\n", pc.pivot(index="city", columns="year", values="n"))
print("\nхотын дундаж accuracy:\n", pc.groupby("city")["acc"].mean().sort_values().round(3))


           city  year    n  approve_rate    acc  acc_gender  gender_gain
0        Дархан  2024  200         0.430  0.840       0.845        0.005
1        Дархан  2025  200         0.480  0.810       0.810        0.000
2        Дархан  2026  200         0.540  0.870       0.860       -0.010
3   Улаанбаатар  2024  300         0.523  0.887       0.887        0.000
4   Улаанбаатар  2025  300         0.443  0.900       0.893       -0.007
5   Улаанбаатар  2026  300         0.453  0.893       0.897        0.003
6          Ховд  2024   90         0.533  0.622       0.689        0.067
7          Ховд  2025   90         0.522  0.622       0.600       -0.022
8          Ховд  2026   90         0.489  0.711       0.656       -0.056
9       Эрдэнэт  2024  200         0.500  0.845       0.850        0.005
10      Эрдэнэт  2025  200         0.500  0.795       0.845        0.050
11      Эрдэнэт  2026  200         0.460  0.855       0.910        0.055

accuracy:
 year          2024   2025   2026
city  

**Хариулт (3, 4).** Ховд хамгийн сул (0.652), cell бүрт 90 мөр л байгаа тул test хэсэг жижиг, accuracy их хэлбэлзэнэ. Эрдэнэт-д gender нэмэхэд accuracy өссөн (+0.05), Ховд-д +0.067, -0.022, -0.056 гэж тогтворгүй. Gender нөлөөтэй байгаа нь өмнөх шийдвэрүүд хүйсээр ялгаварласан байж болзошгүй тул fairness-ийг анхаарах хэрэгтэй.

## Тайлан бичих заавар

Тайланг **`Lab9_Тайлан_загвар.docx`** (Лабораторийн тайлангийн загвар №9) файл дээр бичнэ. Загварын 5 хэсэг дээр энэ ноутбукийн агуулга дараах байдлаар тусна.

| Тайлангийн хэсэг | Энэ лабораторид юу бичих вэ? |
|---|---|
| 1. Зорилго | Лабын зорилго (cell, cuboid, hierarchy, measure; cube-ийн тооцоолол; cube space дахь mining). Хүрсэн эсэхээ 5-р хэсэгт бичнэ |
| 2. Онол | Лекцийн слайдын ойлголтуудыг өөрийн үгээр товч (cell, cuboid, ∏(Lᵢ + 1), concept hierarchy, measure-ийн 3 төрөл, materialization, Iceberg, BUC, closed cell, ROLAP/MOLAP, SelfExp/InExp, prediction cube) |
| 3. Аргачлал | Ноутбукийн **чухал 3-5 код** (жишээ нь `cube_groupby`, `buc`, `is_closed`, SelfExp/InExp, `cell_acc`) ба тус бүр ямар онолыг хэрхэн хэрэгжүүлснийг 2-3 өгүүлбэрээр |
| 4. Үр дүн | TASK 1-13-ын үр дүнгийн хүснэгт (загварт бэлэн), тохиромжтой график, хүснэгтийн зураг |
| 5. Дүгнэлт | Юу ойлгосон, юу хэцүү байсан, яаж шийдсэн. Доорх асуултуудад хариулна |
| Хавсралт | A. `Lab9_CubeSpace.ipynb`, `cs_*.csv`; B. git log; C. Python хувилбар, сангууд; D. GitHub холбоос |

### Дүгнэлт хэсгийн асуултууд
1. Аль даалгавар хамгийн хэцүү байсан бэ? Яаж шийдсэн бэ?
2. Antimonotonicity-г өөрийн үгээр тайлбарла. BUC-д pruning яагаад зөв байдаг вэ?
3. Cube-ийг бүгдийг нь тооцохын оронд Iceberg, closed cell, partial materialization ашиглах нь яагаад хэрэгтэй вэ? (3-5 өгүүлбэр)
4. Өөрийн салбарт (борлуулалт, сургалт, эрүүл мэнд гэх мэт) discovery-driven exploration-г хэрхэн хэрэглэх вэ?